# Suite1 Project3 Regression

**Dataset:** California Housing

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 3: Linear Regression & Model Evaluation"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('housing.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite1_project3_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 3: LINEAR REGRESSION & MODEL EVALUATION

In [ ]:
X = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']]
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]:,}, Test size: {X_test.shape[0]:,}")

# ── 1. Simple Linear Regression ──

## 1. Simple Linear Regression — MedInc → MedHouseVal

In [ ]:
X_simple = X_train[['MedInc']]
X_test_simple = X_test[['MedInc']]

model_simple = LinearRegression()
model_simple.fit(X_simple, y_train)
y_pred_simple = model_simple.predict(X_test_simple)

print(f"Equation: MedHouseVal = {model_simple.coef_[0]:.4f} × MedInc + {model_simple.intercept_:.4f}")
print(f"R² (test):  {r2_score(y_test, y_pred_simple):.4f}")
print(f"RMSE:       {np.sqrt(mean_squared_error(y_test, y_pred_simple)):.4f}")
print(f"MAE:        {mean_absolute_error(y_test, y_pred_simple):.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_test_simple, y_test, alpha=0.2, s=5, label='Actual', color='#2e86de')
axes[0].plot(X_test_simple.sort_values(by='MedInc'), 
             model_simple.predict(X_test_simple.sort_values(by='MedInc')), 
             color='#e74c3c', linewidth=2, label='Regression Line')
axes[0].set_xlabel('Median Income ($10K)', fontsize=12)
axes[0].set_ylabel('Median House Value ($100K)', fontsize=12)
axes[0].set_title('Simple Linear Regression', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals_simple = y_test - y_pred_simple
axes[1].scatter(y_pred_simple, residuals_simple, alpha=0.2, s=5, color='#2e86de')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Value ($100K)', fontsize=12)
axes[1].set_ylabel('Residuals', fontsize=12)
axes[1].set_title('Residual Plot — Simple Model', fontsize=13)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
# 'p3_simple_regression')
plt.show()
print("[Saved] p3_simple_regression.png")

# ── 2. Multiple Linear Regression ──

## 2. Multiple Linear Regression — All Features

In [ ]:
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)

r2_multi = r2_score(y_test, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_multi))
mae_multi = mean_absolute_error(y_test, y_pred_multi)

print(f"R² (test):  {r2_multi:.4f}")
print(f"RMSE:       {rmse_multi:.4f}")
print(f"MAE:        {mae_multi:.4f}")
print(f"\nCoefficients:")
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model_multi.coef_}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

# ── 3. Model Comparison ──

## 3. Model Comparison: Simple vs Multiple

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['R²', 'RMSE', 'MAE'],
    'Simple (Income Only)': [r2_score(y_test, y_pred_simple), 
                             np.sqrt(mean_squared_error(y_test, y_pred_simple)),
                             mean_absolute_error(y_test, y_pred_simple)],
    'Multiple (All Features)': [r2_multi, rmse_multi, mae_multi]
})
print(comparison.round(4).to_string(index=False))

# ── 4. Residual Analysis ──

## 4. Residual Analysis (Multiple Regression)

In [ ]:
residuals_multi = y_test - y_pred_multi

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted
axes[0,0].scatter(y_pred_multi, residuals_multi, alpha=0.2, s=5, color='#2e86de')
axes[0,0].axhline(y=0, color='red', linestyle='--')
axes[0,0].set_xlabel('Predicted Value ($100K)')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Predicted')
axes[0,0].grid(True, alpha=0.3)

# Histogram of Residuals
axes[0,1].hist(residuals_multi, bins=50, color='#2e86de', edgecolor='white', alpha=0.7)
axes[0,1].axvline(x=0, color='red', linestyle='--')
axes[0,1].set_xlabel('Residual')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Histogram of Residuals')

# Q-Q plot
stats.probplot(residuals_multi, dist="norm", plot=axes[1,0])
axes[1,0].set_title('Q-Q Plot (Normality Check)')
axes[1,0].grid(True, alpha=0.3)

# Actual vs Predicted
axes[1,1].scatter(y_test, y_pred_multi, alpha=0.2, s=5, color='#2e86de')
axes[1,1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1,1].set_xlabel('Actual Values ($100K)')
axes[1,1].set_ylabel('Predicted Values ($100K)')
axes[1,1].set_title(f'Actual vs Predicted (R² = {r2_multi:.4f})')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
# 'p3_residual_analysis')
plt.show()
print("[Saved] p3_residual_analysis.png")

# ── 5. Adding complexity: Polynomial / Regularized ──

## 5. Extended: Ridge & Lasso Comparison

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    r2_r = r2_score(y_test, ridge.predict(X_test_scaled))
    
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train_scaled, y_train)
    r2_l = r2_score(y_test, lasso.predict(X_test_scaled))
    
    results.append({'Alpha': alpha, 'Ridge R²': round(r2_r,4), 'Lasso R²': round(r2_l,4),
                    'Lasso Zero Features': (np.abs(lasso.coef_) < 1e-10).sum()})

print(pd.DataFrame(results).to_string(index=False))

# Export
results_final = {
    'simple_r2': round(r2_score(y_test, y_pred_simple), 4),
    'simple_rmse': round(np.sqrt(mean_squared_error(y_test, y_pred_simple)), 4),
    'multi_r2': round(r2_multi, 4),
    'multi_rmse': round(rmse_multi, 4),
    'multi_mae': round(mae_multi, 4),
    'coefficients': coef_df.to_dict('records'),
    'ridge_lasso_comparison': results,
    'mean_residual': float(np.mean(residuals_multi)),
    'std_residual': float(np.std(residuals_multi)),
}
    json.dump(results_final, f, indent=2, default=str)

print("Done.")